In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, HTML, HBox, Layout
from IPython.display import display

# ============================================================
# ALL-POLE IIR LATTICE IMPLEMENTATION — SINGLE CANVAS
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'
MAX_STAGES = 4
N = 100

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.iirlat-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.iirlat-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.iirlat-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.iirlat-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.iirlat-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.iirlat-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.iirlat-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="iirlat-root">

<div class="iirlat-header">
All-Pole IIR Lattice Implementation — Reversing the FIR Structure
</div>

<div class="iirlat-doc">

An all-pole IIR lattice uses the same reflection coefficients
K₁, K₂, ... that define the corresponding FIR lattice polynomial

<div class="iirlat-equation">
<b>
A<sub>N</sub>(z)
=
1 + α<sub>N</sub>[1]z<sup>−1</sup>
+ ... +
α<sub>N</sub>[N]z<sup>−N</sup>.
</b>
</div>

The all-pole system is

<div class="iirlat-equation">
<b>
H(z) = 1 / A<sub>N</sub>(z).
</b>
</div>

The forward variables are calculated in the reverse stage direction:

<div class="iirlat-equation">
<b>
f<sub>N</sub>[n] = x[n]
</b>
</div>

<div class="iirlat-equation">
<b>
f<sub>m−1</sub>[n]
=
f<sub>m</sub>[n]
−
K<sub>m</sub>g<sub>m−1</sub>[n−1].
</b>
</div>

The backward variables satisfy

<div class="iirlat-equation">
<b>
g<sub>m</sub>[n]
=
K<sub>m</sub>f<sub>m−1</sub>[n]
+
g<sub>m−1</sub>[n−1].
</b>
</div>

Finally,

<div class="iirlat-equation">
<b>
y[n] = f₀[n] = g₀[n].
</b>
</div>

<div class="iirlat-equation">
<b>
FIR lattice A<sub>N</sub>(z)
→ reverse realization
→ all-pole IIR 1/A<sub>N</sub>(z)
</b>
</div>

Only the reflection coefficients belonging to active stages are enabled.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

stage_slider = IntSlider(value=4,min=1,max=MAX_STAGES,step=1,description='Stages:',continuous_update=True,style={'description_width':'45px'},layout=Layout(width='170px'))

K1_slider = FloatSlider(value=0.35,min=-0.85,max=0.85,step=0.05,description='K₁:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K2_slider = FloatSlider(value=-0.30,min=-0.85,max=0.85,step=0.05,description='K₂:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K3_slider = FloatSlider(value=0.25,min=-0.85,max=0.85,step=0.05,description='K₃:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K4_slider = FloatSlider(value=-0.20,min=-0.85,max=0.85,step=0.05,description='K₄:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K_sliders = [K1_slider,K2_slider,K3_slider,K4_slider]

controls = HBox([stage_slider,K1_slider,K2_slider,K3_slider,K4_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 8px',margin='0 0 5px 0'))

# ============================================================
# TEST INPUT
# ============================================================

n = np.arange(N)

x = np.zeros(N)

x[0] = 1.0

x += 0.22*np.sin(0.18*np.pi*n)
x += 0.12*np.sin(0.55*np.pi*n)

# ============================================================
# COEFFICIENT FUNCTIONS
# ============================================================

def get_reflection_coefficients():

    return np.array([K1_slider.value,K2_slider.value,K3_slider.value,K4_slider.value])

def reflection_to_polynomial(K):

    A = np.array([1.0])
    B = np.array([1.0])

    A_history = [A.copy()]
    B_history = [B.copy()]

    for Km in K:

        A_pad = np.append(A,0.0)
        B_shift = np.insert(B,0,0.0)

        A_new = A_pad+Km*B_shift
        B_new = Km*A_pad+B_shift

        A = A_new
        B = B_new

        A_history.append(A.copy())
        B_history.append(B.copy())

    return A,B,A_history,B_history

# ============================================================
# IIR LATTICE
# ============================================================

def iir_lattice_filter(x,K):

    M = len(K)

    f_history = np.zeros((M+1,len(x)))
    g_history = np.zeros((M+1,len(x)))

    previous_g = np.zeros(M+1)

    for sample in range(len(x)):

        f_current = np.zeros(M+1)
        g_current = np.zeros(M+1)

        f_current[M] = x[sample]

        for m in range(M,0,-1):

            f_current[m-1] = f_current[m]-K[m-1]*previous_g[m-1]

        g_current[0] = f_current[0]

        for m in range(1,M+1):

            g_current[m] = K[m-1]*f_current[m-1]+previous_g[m-1]

        f_history[:,sample] = f_current
        g_history[:,sample] = g_current

        previous_g = g_current.copy()

    return f_history[0].copy(),f_history,g_history

# ============================================================
# PRECOMPUTE TIME-DOMAIN LIMITS
# ============================================================

test_values = [-0.85,0.85]

all_internal_values = []
all_output_values = []

for K1 in test_values:

    for K2 in test_values:

        for K3 in test_values:

            for K4 in test_values:

                K_test = np.array([K1,K2,K3,K4])

                A_test,B_test,A_hist_test,B_hist_test = reflection_to_polynomial(K_test)

                y_lattice_test,f_hist_test,g_hist_test = iir_lattice_filter(x,K_test)

                y_direct_test = signal.lfilter([1.0],A_test,x)

                for m in range(MAX_STAGES+1):

                    all_internal_values.extend(f_hist_test[m])
                    all_internal_values.extend(g_hist_test[m])

                all_output_values.extend(y_lattice_test)
                all_output_values.extend(y_direct_test)

internal_limit = 1.10*max(np.max(np.abs(all_internal_values)),1.0)

output_limit = 1.10*max(np.max(np.abs(all_output_values)),1.0)

# ============================================================
# INITIAL VALUES
# ============================================================

active_stages = stage_slider.value

K_all = get_reflection_coefficients()

K = K_all[:active_stages]

A,B,A_history,B_history = reflection_to_polynomial(K)

y_lattice,f_history,g_history = iir_lattice_filter(x,K)

y_direct = signal.lfilter([1.0],A,x)

difference = y_direct-y_lattice

omega,H_direct = signal.freqz([1.0],A,worN=1024)

poles = np.roots(A)

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# SINGLE FIGURE / SINGLE CANVAS
# ============================================================

fig = plt.figure(figsize=(9.0,14.0))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

gs = fig.add_gridspec(5,2,height_ratios=[1.10,1.00,1.05,0.95,1.00],hspace=0.55,wspace=0.30)

ax_structure = fig.add_subplot(gs[0,:])

ax_coefficients = fig.add_subplot(gs[1,0])

ax_response = fig.add_subplot(gs[1,1])

ax_poles = fig.add_subplot(gs[2,0])

ax_forward = fig.add_subplot(gs[2,1])

ax_backward = fig.add_subplot(gs[3,:])

ax_output = fig.add_subplot(gs[4,0])

ax_difference = fig.add_subplot(gs[4,1])

# ============================================================
# STRUCTURE
# ============================================================

ax_structure.set_xlim(0,11)
ax_structure.set_ylim(-2.0,2.0)
ax_structure.axis('off')
ax_structure.set_title('All-Pole IIR Lattice Structure')

stage_centers = [2.4,4.5,6.6,8.7]

stage_rectangles = []
stage_labels = []
upper_labels = []
lower_labels = []

ax_structure.text(10.35,0.98,r'$f_N[n]=x[n]$',fontsize=10.5,fontweight='bold',ha='center')

ax_structure.text(0.55,0.98,r'$f_0[n]=y[n]$',fontsize=10.5,fontweight='bold',ha='center')

ax_structure.text(0.55,-0.98,r'$g_0[n]=y[n]$',fontsize=10.5,fontweight='bold',ha='center')

for m,xc in enumerate(stage_centers):

    rect = plt.Rectangle((xc-0.72,-1.35),1.44,2.70,fill=False,linewidth=1.3)

    ax_structure.add_patch(rect)

    stage_rectangles.append(rect)

    label = ax_structure.text(xc,1.55,f'Stage {m+1}',ha='center',fontsize=10.5,fontweight='bold')

    stage_labels.append(label)

    upper = ax_structure.text(xc,0.30,'',ha='center',va='center',fontsize=10)

    lower = ax_structure.text(xc,-0.30,'',ha='center',va='center',fontsize=10)

    upper_labels.append(upper)
    lower_labels.append(lower)

    ax_structure.annotate('',xy=(xc-0.55,0.75),xytext=(xc+0.55,0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(xc+0.55,-0.75),xytext=(xc-0.55,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(xc-0.52,0.73),xytext=(xc-0.52,-0.73),arrowprops={'arrowstyle':'->','linewidth':1.0})

    ax_structure.annotate('',xy=(xc+0.52,-0.73),xytext=(xc-0.48,0.73),arrowprops={'arrowstyle':'->','linewidth':1.0})

    ax_structure.text(xc-0.18,-1.10,r'$z^{-1}$',ha='center',fontsize=9.5)

for m in range(MAX_STAGES-1):

    left_x = stage_centers[m]
    right_x = stage_centers[m+1]

    ax_structure.annotate('',xy=(left_x+0.72,0.75),xytext=(right_x-0.72,0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(right_x-0.72,-0.75),xytext=(left_x+0.72,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.2})

ax_structure.annotate('',xy=(9.42,0.75),xytext=(10.15,0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.annotate('',xy=(1.68,0.75),xytext=(0.90,0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.annotate('',xy=(1.68,-0.75),xytext=(0.90,-0.75),arrowprops={'arrowstyle':'->','linewidth':1.3})

# ============================================================
# DENOMINATOR COEFFICIENTS
# ============================================================

coefficient_indices = np.arange(MAX_STAGES+1)

initial_coefficients = np.zeros(MAX_STAGES+1)

initial_coefficients[:len(A)] = A

coefficient_stems = ax_coefficients.vlines(coefficient_indices,0,initial_coefficients,linewidth=1.4)

coefficient_markers, = ax_coefficients.plot(coefficient_indices,initial_coefficients,'o',markersize=5)

ax_coefficients.axhline(0,linewidth=0.8)

ax_coefficients.set_xlim(-0.5,MAX_STAGES+0.5)

ax_coefficients.set_ylim(-2.5,2.5)

ax_coefficients.set_xticks(coefficient_indices)

ax_coefficients.set_title('Equivalent Direct-Form Denominator')

ax_coefficients.set_xlabel('Coefficient index k')

ax_coefficients.set_ylabel(r'$\alpha_N[k]$')

ax_coefficients.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

response_line, = ax_response.plot(omega/np.pi,np.abs(H_direct),linewidth=1.4)

ax_response.set_xlim(0,1)

ax_response.set_ylim(0,max(1.0,1.10*np.max(np.abs(H_direct))))

ax_response.set_title('All-Pole IIR Frequency Response')

ax_response.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_response.set_ylabel(r'$|H(e^{j\omega})|$')

ax_response.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# POLES
# ============================================================

theta = np.linspace(0,2*np.pi,400)

ax_poles.plot(np.cos(theta),np.sin(theta),'--',linewidth=0.9)

ax_poles.axhline(0,linewidth=0.8)

ax_poles.axvline(0,linewidth=0.8)

pole_markers, = ax_poles.plot(np.real(poles),np.imag(poles),'x',markersize=8,markeredgewidth=1.6)

ax_poles.set_xlim(-1.1,1.1)

ax_poles.set_ylim(-1.1,1.1)

ax_poles.set_aspect('equal',adjustable='box')

ax_poles.set_title('Pole Locations')

ax_poles.set_xlabel('Real')

ax_poles.set_ylabel('Imaginary')

ax_poles.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# FORWARD SIGNALS
# ============================================================

forward_lines = []

for m in range(MAX_STAGES+1):

    data = f_history[m] if m <= active_stages else np.full(N,np.nan)

    line, = ax_forward.plot(n,data,linewidth=1.05,label=rf'$f_{m}[n]$')

    forward_lines.append(line)

ax_forward.set_xlim(0,N-1)

ax_forward.set_ylim(-internal_limit,internal_limit)

ax_forward.set_title('Internal Forward Signals')

ax_forward.set_xlabel('Sample index n')

ax_forward.set_ylabel(r'$f_m[n]$')

ax_forward.grid(True,linestyle=':',alpha=0.30)

ax_forward.legend(loc='upper right',ncol=2)

# ============================================================
# BACKWARD SIGNALS
# ============================================================

backward_lines = []

for m in range(MAX_STAGES+1):

    data = g_history[m] if m <= active_stages else np.full(N,np.nan)

    line, = ax_backward.plot(n,data,linewidth=1.05,label=rf'$g_{m}[n]$')

    backward_lines.append(line)

ax_backward.set_xlim(0,N-1)

ax_backward.set_ylim(-internal_limit,internal_limit)

ax_backward.set_title('Internal Backward Signals')

ax_backward.set_xlabel('Sample index n')

ax_backward.set_ylabel(r'$g_m[n]$')

ax_backward.grid(True,linestyle=':',alpha=0.30)

ax_backward.legend(loc='upper right',ncol=5)

# ============================================================
# OUTPUT COMPARISON
# ============================================================

direct_line, = ax_output.plot(n,y_direct,linewidth=1.5,label='Direct all-pole IIR')

lattice_line, = ax_output.plot(n,y_lattice,'--',linewidth=1.3,label='IIR lattice')

ax_output.set_xlim(0,N-1)

ax_output.set_ylim(-output_limit,output_limit)

ax_output.set_title('Final Output Comparison')

ax_output.set_xlabel('Sample index n')

ax_output.set_ylabel('y[n]')

ax_output.grid(True,linestyle=':',alpha=0.30)

ax_output.legend(loc='upper right')

# ============================================================
# DIFFERENCE
# ============================================================

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_ylim(-1e-12,1e-12)

ax_difference.set_title('Numerical Difference')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel(r'$y_D[n]-y_L[n]$')

ax_difference.grid(True,linestyle=':',alpha=0.30)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    active_stages = stage_slider.value

    for i,slider in enumerate(K_sliders):

        slider.disabled = i >= active_stages

    K_all = get_reflection_coefficients()

    K = K_all[:active_stages]

    A,B,A_history,B_history = reflection_to_polynomial(K)

    y_lattice,f_history,g_history = iir_lattice_filter(x,K)

    y_direct = signal.lfilter([1.0],A,x)

    difference = y_direct-y_lattice

    for m in range(MAX_STAGES):

        Km = K_all[m]

        upper_labels[m].set_text(rf'$K_{m+1}={Km:.2f}$')

        lower_labels[m].set_text(rf'$K_{m+1}={Km:.2f}$')

        alpha = 1.0 if m < active_stages else 0.18

        stage_rectangles[m].set_alpha(alpha)

        stage_labels[m].set_alpha(alpha)

        upper_labels[m].set_alpha(alpha)

        lower_labels[m].set_alpha(alpha)

    coefficients = np.zeros(MAX_STAGES+1)

    coefficients[:len(A)] = A

    coefficient_stems.set_segments([[(k,0),(k,coefficients[k])] for k in range(MAX_STAGES+1)])

    coefficient_markers.set_ydata(coefficients)

    omega,H = signal.freqz([1.0],A,worN=1024)

    response_line.set_ydata(np.abs(H))

    ax_response.set_ylim(0,max(1.0,1.10*np.max(np.abs(H))))

    poles = np.roots(A)

    pole_markers.set_data(np.real(poles),np.imag(poles))

    for m in range(MAX_STAGES+1):

        if m <= active_stages:

            forward_lines[m].set_ydata(f_history[m])

            backward_lines[m].set_ydata(g_history[m])

        else:

            forward_lines[m].set_ydata(np.full(N,np.nan))

            backward_lines[m].set_ydata(np.full(N,np.nan))

    direct_line.set_ydata(y_direct)

    lattice_line.set_ydata(y_lattice)

    difference_line.set_ydata(difference)

    maximum_difference = np.max(np.abs(difference))

    maximum_pole_radius = np.max(np.abs(poles))

    K_text = ', '.join([f'K{i+1} = {K[i]:.2f}' for i in range(active_stages)])

    A_text = ', '.join([f'{value:.6f}' for value in A])

    stage_rows = ""

    for m in range(1,active_stages+1):

        coefficient_text = ', '.join([f'{value:.5f}' for value in A_history[m]])

        stage_rows += f"""
        <tr>
        <td style="padding:2px 10px;"><b>m = {m}</b></td>
        <td style="padding:2px 10px;">K<sub>{m}</sub> = {K[m-1]:.3f}</td>
        <td style="padding:2px 10px;">A<sub>{m}</sub> = [{coefficient_text}]</td>
        </tr>
        """

    result_html.value = f"""
    <div class="iirlat-root">

    <div class="iirlat-box iirlat-result">

    <div class="iirlat-title">
    Current all-pole IIR lattice implementation
    </div>

    <b>Active stages:</b> {active_stages}

    &nbsp;&nbsp;&nbsp;

    <b>Active reflection coefficients:</b> {K_text}

    <br><br>

    Equivalent direct-form denominator:

    <div class="iirlat-equation">
    A(z) = [{A_text}]
    </div>

    Maximum pole radius:

    <b>{maximum_pole_radius:.6f}</b>

    &nbsp;&nbsp;&nbsp;

    Maximum
    |y<sub>direct</sub>[n] − y<sub>lattice</sub>[n]|:

    <b>{maximum_difference:.3e}</b>

    <table style="margin-top:7px;font-size:12.5px;border-collapse:collapse;">
    {stage_rows}
    </table>

    </div>

    </div>
    """

    fig.canvas.draw()

# ============================================================
# OBSERVERS
# ============================================================

stage_slider.observe(update,names='value')

K1_slider.observe(update,names='value')

K2_slider.observe(update,names='value')

K3_slider.observe(update,names='value')

K4_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(controls)

display(fig.canvas)

update()